# 01 — Exploratory data analysis (Jurkat BBBC048, Ch3 + Ch6)

Exploration only. Every figure is produced by functions in `scripts/eda.py` (also runnable as `python -m scripts.eda`),
so `figures/` is still written only by code in `scripts/`.

No split exists yet (`splits/splits.json` is missing), so this looks at **all cells**. Nothing here is fitted or tuned.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
from IPython.display import display

from src import config
from src.data import load_dataset
from scripts import eda

X, y7, cell_ids = load_dataset(7)
_, y4, _ = load_dataset(4)
X.shape, X.dtype

## 1. Class counts (7 and 4 classes)

In [ ]:
counts = eda.class_count_table(y7)
display(counts)

In [ ]:
# 4-class view straight from the loader, as a cross-check of the merge
display(counts.groupby("class_4", sort=False)[["n_7", "pct_7"]].sum()
        .rename(columns={"n_7": "n_4", "pct_7": "pct_4"}))
assert np.bincount(y4).tolist() == counts.groupby("class_4", sort=False)["n_7"].sum().tolist()

## 2. Example cells: 8 random cells per phase (seed 0), Ch3 and Ch6
The same cells appear in both grids, so you can compare them column by column.

In [ ]:
picks = eda.pick_examples(y7, np.random.default_rng(eda.SEED))
for c, ch in enumerate(config.INPUT_CHANNELS):
    fig = eda.plot_examples(X, cell_ids, picks, c)
    eda.save(fig, f"eda_examples_{ch}.png")

## 3. Pixel-intensity histogram per channel

In [ ]:
fig = eda.plot_intensity_histograms(X, y7)
eda.save(fig, "eda_intensity_hist.png")

Follow-up: is the class difference in Ch6 in the cell or in the background? Median of each cell's 5-px border ring, then the median per phase.

In [ ]:
display(eda.background_table(X, y7))

## 4. Image-size check (reads every raw Ch3/Ch6 file header, not the cache)

In [ ]:
sizes, missing = eda.image_size_check()
display(sizes)
print(f"missing channel files: {len(missing)}")
assert len(missing) == 0
assert (sizes[["width", "height"]] == config.IMAGE_SIZE).all().all() and (sizes["mode"] == "L").all()